# Resampling and Affines

STARE allows for resampling the original PET image to speed up the clustering
process. This notebook explores what that resampling looks like and demonstrates
that the resampled image is in the same location as the original.

In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt
from nilearn import image, plotting
import numpy as np


In [ ]:
""" Locate and load a PET image.
    Adapt the paths directly below to find an image you can access.
"""

subject = "NHFDG911"
data_path = Path("/home/mike/cache/cerepet_data")
example_image_path = data_path / subject / "moco" / f"{subject}.05.MCFI.hdr"

print(f"Image {'exists' if example_image_path.exists() else'does not exist'}.")
orig_img = image.load_img(example_image_path)
print(type(orig_img), orig_img.shape)
print(orig_img.affine)

disp_01 = plotting.plot_epi(orig_img, title="Original image")


In [ ]:
""" Resample that image to 3mm isotropic resolution. """

new_3mm_img = image.resample_img(
    orig_img, target_affine=np.diag((3, 3, 3))
)
print(type(new_3mm_img), new_3mm_img.shape)
print(new_3mm_img.affine)

disp_02 = plotting.plot_epi(new_3mm_img, title="3mm isotropic image")


In [ ]:
""" Resample that image to half its original resolution. """

new_affine = orig_img.affine[:3, :3]
new_2x_img = image.resample_img(
    orig_img, target_affine=new_affine * 2.0
)
print(type(new_2x_img), new_2x_img.shape)
print(new_2x_img.affine)

disp_03 = plotting.plot_epi(new_2x_img, title="Half-size image")


In [ ]:
""" Create a single figure so we can plot all three
    images side-by-side for easier comparison.
    The vmin/vmax and cut_coords values were manually selected from
    the example image and coordinated across images for comparability.
"""

fig, axes = plt.subplots(nrows=3, figsize=(8, 10))
plotting.plot_epi(orig_img, title="Original image", vmin=0.0, vmax=10000.0,
                  cut_coords=(-10, 10, 70), colorbar=False, axes=axes[0])
plotting.plot_epi(new_3mm_img, title="3mm iso image", vmin=0.0, vmax=10000.0,
                  cut_coords=(-10, 10, 70), colorbar=False, axes=axes[1])
plotting.plot_epi(new_2x_img, title="Half-size image", vmin=0.0, vmax=10000.0,
                  cut_coords=(-10, 10, 70), colorbar=False, axes=axes[2])


In [ ]:
fig.savefig(data_path / f"{subject}_in_three_resolutions.png")
